# Hyperparameter Tuning — Logistic Regression, Random Forest & XGBoost

**Approach:** Due to the size of the dataset (1,000,000 rows), exhaustive GridSearchCV on the full training set is computationally impractical. `RandomizedSearchCV` is instead applied on a stratified subsample of the training data (preserving the ~1.1% fraud rate) to identify strong hyperparameter combinations for all three models — Logistic Regression, Random Forest, and XGBoost — so the tuning approach is consistent across the comparison. The selected best parameters are then used to refit each model on the **full** training set for final evaluation.

## 1. Install & Import Libraries

Install and import the libraries required for hyperparameter optimisation. RandomizedSearchCV is used to explore a limited number of parameter combinations rather than testing every possible combination reducing computational cost. StratifiedKFold preserves the fraud class distribution during cross-validation while the imbalanced-learn Pipeline allows preprocessing steps such as SMOTE to be applied separately within each training fold.

In [1]:
!pip install xgboost imbalanced-learn -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score, classification_report

print('Libraries loaded ✅')

Libraries loaded ✅


## 2. Load & Preprocess Dataset (same as model notebooks)

Loads the Bank Account Fraud dataset and applies the same preprocessing procedure used in the individual model notebooks. Invalid negative values are corrected, categorical variables are encoded numerically and the target variable fraud_bool is separated from the predictor variables.

The dataset is then divided into the same 80% training and 20% test split using stratified sampling and random_state=42. Maintaining the same split ensures that the tuned models can be compared fairly with the original baseline models.

In [2]:
df = pd.read_csv('Base.csv')
print(f'Shape: {df.shape}')

# Fix negative values
df['session_length_in_minutes'] = df['session_length_in_minutes'].clip(lower=0)
df['device_distinct_emails_8w'] = df['device_distinct_emails_8w'].clip(lower=0)

# Encode categorical columns
cat_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop('fraud_bool', axis=1)
y = df['fraud_bool']

# Same 80/20 stratified split, same seed, as the model notebooks
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set    : {X_test.shape[0]:,} rows')

Shape: (1000000, 32)
Training set: 800,000 rows
Test set    : 200,000 rows


## 3. Create Stratified Subsample for Search

Hyperparameter tuning on the entire training dataset would require substantial computational time because the BAF dataset contains approximately one million observations. Therefore, a stratified subsample of 80,000 observations is selected from the training set for the search process.

Stratification preserves approximately the same fraud rate as the full dataset ensuring that the search sample remains representative of the class imbalance. The test set is not used during hyperparameter search and remains reserved for final evaluation.

In [3]:
# Stratified subsample of the TRAINING set only — keeps the ~1.1% fraud rate.
# 80,000 rows keeps the search fast while still giving 880 fraud examples to learn from.
SEARCH_SAMPLE_SIZE = 80000

X_search, _, y_search, _ = train_test_split(
    X_train, y_train,
    train_size=SEARCH_SAMPLE_SIZE,
    random_state=42,
    stratify=y_train
)

print(f'Search subsample: {X_search.shape[0]:,} rows')
print(f'  Non-Fraud: {(y_search==0).sum():,}')
print(f'  Fraud    : {(y_search==1).sum():,} ({(y_search==1).mean()*100:.2f}%)')

Search subsample: 80,000 rows
  Non-Fraud: 79,118
  Fraud    : 882 (1.10%)


## 4. Logistic Regression — Hyperparameter Search

Searches for an improved Logistic Regression configuration using RandomizedSearchCV. The main parameters explored include the regularisation strength (C), penalty type and class weighting.

Standardisation and SMOTE are included inside an imbalanced-learn Pipeline. This is important because both preprocessing steps are fitted separately within each cross-validation training fold rather than being applied before cross-validation. This prevents information from the validation fold from influencing the training process and reduces the risk of data leakage.

Average Precision is used as the optimisation metric because the fraud class represents only a small proportion of the dataset and Precision-Recall-based measures are particularly informative under severe class imbalance.

In [4]:
from imblearn.pipeline import Pipeline as ImbPipeline

# Scaling AND SMOTE are now inside the pipeline, so RandomizedSearchCV
# fits both fresh on each training fold only — no leakage into the
# validation fold. Pass the RAW (unscaled, unresampled) search subsample.
lr_param_dist = {
    'clf__C'            : [0.01, 0.1, 1, 10, 100],
    'clf__penalty'       : ['l1', 'l2'],
    'clf__solver'        : ['liblinear'],   # liblinear supports both l1 and l2
    'clf__class_weight'  : [None, 'balanced']
}

lr_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_dist,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Running RandomizedSearchCV for Logistic Regression (10 candidates x 3 folds)...')
lr_search.fit(X_search, y_search)  # raw subsample — scaler + SMOTE run per fold inside the pipeline

print('\nBest LR parameters:')
print(lr_search.best_params_)
print(f'Best CV ROC-AUC: {lr_search.best_score_:.4f}')

Running RandomizedSearchCV for Logistic Regression (10 candidates x 3 folds)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best LR parameters:
{'clf__solver': 'liblinear', 'clf__penalty': 'l1', 'clf__class_weight': None, 'clf__C': 0.01}
Best CV ROC-AUC: 0.8500


## 5. Logistic Regression — Refit on Full Training Set

After identifying the best Logistic Regression hyperparameters using the search subsample, the selected configuration is retrained using the full training dataset.

StandardScaler is fitted only on the full training set and then applied to both the training and test sets. SMOTE is subsequently applied to the scaled training data only. At this stage, applying SMOTE before fitting the final model does not create cross-validation leakage because the hyperparameter search has already been completed and the test set remains untouched.

The final tuned model is then evaluated using the held-out test set to determine whether tuning improves performance compared with the original baseline Logistic Regression model.

In [5]:
# Scale the FULL training and test sets (fit scaler on training data only)
scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train)
X_test_scaled = scaler_full.transform(X_test)

# Apply SMOTE on the FULL scaled training set for the final refit.
print('Applying SMOTE to full scaled training set...')
smote_lr_full = SMOTE(random_state=42, k_neighbors=5)
X_train_lr_sm, y_train_lr_sm = smote_lr_full.fit_resample(X_train_scaled, y_train)

# best_params_ keys are prefixed with 'clf__' because the search used a Pipeline
lr_best_params = {k.replace('clf__', ''): v for k, v in lr_search.best_params_.items()}

lr_best = LogisticRegression(
    **lr_best_params,
    max_iter=1000,
    random_state=42
)

print('Refitting tuned Logistic Regression on full training set...')
lr_best.fit(X_train_lr_sm, y_train_lr_sm)

y_pred_lr = lr_best.predict(X_test_scaled)
y_prob_lr = lr_best.predict_proba(X_test_scaled)[:, 1]

print('\nTUNED LOGISTIC REGRESSION — TEST SET RESULTS')
print('=' * 55)
print(classification_report(y_test, y_pred_lr, target_names=['Non-Fraud', 'Fraud']))
print(f'ROC-AUC       : {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'Avg Precision : {average_precision_score(y_test, y_prob_lr):.4f}')

Applying SMOTE to full scaled training set...
Refitting tuned Logistic Regression on full training set...

TUNED LOGISTIC REGRESSION — TEST SET RESULTS
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.80      0.89    197794
       Fraud       0.04      0.77      0.08      2206

    accuracy                           0.80    200000
   macro avg       0.52      0.79      0.48    200000
weighted avg       0.99      0.80      0.88    200000

ROC-AUC       : 0.8626
Avg Precision : 0.1220


## 6. Random Forest — Hyperparameter Search

Performs hyperparameter optimisation for the Random Forest model. The search considers parameters controlling the number of trees, tree depth, minimum samples required for splits and leaves, and the number of features considered at each split.

SMOTE is included inside the Pipeline so that synthetic minority-class samples are generated independently within each cross-validation training fold. This prevents synthetic observations derived from one fold from appearing in another validation fold and avoids the data-leakage problem associated with applying SMOTE before cross-validation.

The search therefore provides a more reliable estimate of which Random Forest configuration generalises best to unseen data.

In [6]:
# SMOTE is now inside the pipeline, so it is fit fresh on each training
# fold only during cross-validation — no leakage into the validation fold.
rf_param_dist = {
    'clf__n_estimators'      : [100, 200, 300],
    'clf__max_depth'         : [8, 10, 12, 15],
    'clf__min_samples_leaf'  : [3, 5, 10, 15],
    'clf__min_samples_split' : [5, 10, 20],
    'clf__max_features'      : ['sqrt', 'log2']
}

rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Running RandomizedSearchCV for Random Forest (20 candidates x 3 folds)...')
rf_search.fit(X_search, y_search)  # raw subsample — SMOTE runs per fold inside the pipeline

print('\nBest RF parameters:')
print(rf_search.best_params_)
print(f'Best CV Average Precision: {rf_search.best_score_:.4f}')

Running RandomizedSearchCV for Random Forest (20 candidates x 3 folds)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best RF parameters:
{'clf__n_estimators': 100, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 15, 'clf__max_features': 'log2', 'clf__max_depth': 15}
Best CV Average Precision: 0.0682


## 7. Random Forest — Refit on Full Training Set

Once the best Random Forest hyperparameters have been identified, the selected model is retrained using the entire training dataset.

SMOTE is applied to the full training set before this final refit because the cross-validation search has already been completed. The tuned Random Forest is then evaluated on the original held-out test set using ROC-AUC, Average Precision, Precision, Recall and F1-score.

In [7]:
# Apply SMOTE on the FULL training set for the final refit
print('Applying SMOTE to full training set...')
smote_full = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote_full.fit_resample(X_train, y_train)

rf_best_params = {k.replace('clf__', ''): v for k, v in rf_search.best_params_.items()}

rf_best = RandomForestClassifier(
    **rf_best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print('Refitting tuned Random Forest on full training set...')
rf_best.fit(X_train_sm, y_train_sm)

y_pred_rf = rf_best.predict(X_test)
y_prob_rf = rf_best.predict_proba(X_test)[:, 1]

print('\nTUNED RANDOM FOREST — TEST SET RESULTS')
print('=' * 55)
print(classification_report(y_test, y_pred_rf, target_names=['Non-Fraud', 'Fraud']))
print(f'ROC-AUC       : {roc_auc_score(y_test, y_prob_rf):.4f}')
print(f'Avg Precision : {average_precision_score(y_test, y_prob_rf):.4f}')

Applying SMOTE to full training set...
Refitting tuned Random Forest on full training set...

TUNED RANDOM FOREST — TEST SET RESULTS
              precision    recall  f1-score   support

   Non-Fraud       0.99      0.93      0.96    197794
       Fraud       0.06      0.42      0.11      2206

    accuracy                           0.93    200000
   macro avg       0.53      0.68      0.54    200000
weighted avg       0.98      0.93      0.95    200000

ROC-AUC       : 0.8349
Avg Precision : 0.0667


## 8. XGBoost — Hyperparameter Search

Tunes the XGBoost model using RandomizedSearchCV. Parameters such as the number of estimators, tree depth, learning rate, subsampling ratio, feature-sampling ratio and scale_pos_weight are explored.

Unlike Logistic Regression and Random Forest, SMOTE is not used for XGBoost. Instead, class imbalance is handled through scale_pos_weight which changes the penalty applied to misclassified fraud observations during training.

Average Precision is used as the search scoring metric because it focuses on performance for the minority fraud class and is more informative than accuracy for this highly imbalanced dataset.

In [8]:
# scale_pos_weight computed on the search subsample
search_scale_pos_weight = (y_search == 0).sum() / (y_search == 1).sum()


xgb_param_dist = {
    'n_estimators'     : [100, 200, 300, 400],
    'max_depth'        : [3, 4, 5, 6, 8],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.2],
    'subsample'        : [0.6, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.8, 1.0],
    'min_child_weight' : [1, 3, 5],
    'scale_pos_weight' : [search_scale_pos_weight * 0.5, search_scale_pos_weight, search_scale_pos_weight * 2]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Running RandomizedSearchCV for XGBoost (30 candidates x 3 folds)...')
xgb_search.fit(X_search, y_search)

print('\nBest XGBoost parameters:')
print(xgb_search.best_params_)
print(f'Best CV Average Precision: {xgb_search.best_score_:.4f}')

Running RandomizedSearchCV for XGBoost (30 candidates x 3 folds)...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best XGBoost parameters:
{'subsample': 0.8, 'scale_pos_weight': np.float64(44.85147392290249), 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best CV Average Precision: 0.1395


## 9. XGBoost — Refit on Full Training Set

After identifying the best XGBoost hyperparameter combination, the selected configuration is retrained using the complete training dataset.

The tuned scale_pos_weight value is retained from the search rather than recalculated separately, ensuring that the final model uses the exact configuration selected by RandomizedSearchCV. Predictions and fraud probabilities are then generated for the held-out test set to evaluate the tuned model's final performance.

The resulting metrics are compared with those from the baseline XGBoost model to determine whether tuning provides a meaningful improvement.

In [9]:

xgb_best = XGBClassifier(
    **xgb_search.best_params_,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

print('Refitting tuned XGBoost on full training set...')
xgb_best.fit(X_train, y_train) 

y_pred_xgb = xgb_best.predict(X_test)
y_prob_xgb = xgb_best.predict_proba(X_test)[:, 1]

print('\nTUNED XGBOOST — TEST SET RESULTS')
print('=' * 55)
print(classification_report(y_test, y_pred_xgb, target_names=['Non-Fraud', 'Fraud']))
print(f'ROC-AUC       : {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(f'Avg Precision : {average_precision_score(y_test, y_prob_xgb):.4f}')

Refitting tuned XGBoost on full training set...

TUNED XGBOOST — TEST SET RESULTS
              precision    recall  f1-score   support

   Non-Fraud       1.00      0.91      0.95    197794
       Fraud       0.08      0.67      0.14      2206

    accuracy                           0.91    200000
   macro avg       0.54      0.79      0.54    200000
weighted avg       0.99      0.91      0.94    200000

ROC-AUC       : 0.8957
Avg Precision : 0.1705


## 10. Before vs After Tuning — Summary Table

Compares the performance of the baseline and tuned versions of Logistic Regression, Random Forest and XGBoost. ROC-AUC, Average Precision and Recall are reported so that changes in overall discrimination and minority-class detection can be examined.

The comparison is used to determine whether the tuned configuration should replace the original model in the final model-comparison notebook. A tuned model is not automatically considered better simply because one metric improves the overall trade-off across the evaluation measures is considered.

In [10]:

summary = pd.DataFrame([
    {'Model': 'Logistic Regression (default)', 'ROC-AUC': 0.8626, 'Avg Precision': 0.1220, 'Recall': 0.7697},
    {'Model': 'Logistic Regression (tuned)',
     'ROC-AUC': round(roc_auc_score(y_test, y_prob_lr), 4),
     'Avg Precision': round(average_precision_score(y_test, y_prob_lr), 4),
     'Recall': round(recall_score(y_test, y_pred_lr), 4)},
    {'Model': 'Random Forest (default)', 'ROC-AUC': 0.8325, 'Avg Precision': 0.0654, 'Recall': 0.3998},
    {'Model': 'Random Forest (tuned)',
     'ROC-AUC': round(roc_auc_score(y_test, y_prob_rf), 4),
     'Avg Precision': round(average_precision_score(y_test, y_prob_rf), 4),
     'Recall': round(recall_score(y_test, y_pred_rf), 4)},
    {'Model': 'XGBoost (default)', 'ROC-AUC': 0.8931, 'Avg Precision': 0.1684, 'Recall': 0.7425},
    {'Model': 'XGBoost (tuned)',
     'ROC-AUC': round(roc_auc_score(y_test, y_prob_xgb), 4),
     'Avg Precision': round(average_precision_score(y_test, y_prob_xgb), 4),
     'Recall': round(recall_score(y_test, y_pred_xgb), 4)},
])

print('BEFORE vs AFTER TUNING — ALL THREE MODELS')
print('=' * 65)
print(summary.to_string(index=False))
print('=' * 65)

BEFORE vs AFTER TUNING — ALL THREE MODELS
                        Model  ROC-AUC  Avg Precision  Recall
Logistic Regression (default)   0.8626         0.1220  0.7697
  Logistic Regression (tuned)   0.8626         0.1220  0.7702
      Random Forest (default)   0.8325         0.0654  0.3998
        Random Forest (tuned)   0.8349         0.0667  0.4211
            XGBoost (default)   0.8931         0.1684  0.7425
              XGBoost (tuned)   0.8957         0.1705  0.6695


Logistic Regression shows little or no meaningful improvement after tuning suggesting that the baseline configuration already provides similar predictive performance.

Random Forest demonstrates modest improvement after tuning particularly in minority-class detection and ranking performance supporting the use of the tuned Random Forest configuration in the final comparison.

For XGBoost, tuning should be interpreted in terms of the trade-off between ROC-AUC, Average Precision, and Recall. If optimisation improves one metric while reducing other important fraud-detection measures, the original configuration may remain preferable. This comparison therefore demonstrates that hyperparameter tuning does not necessarily improve every aspect of model performance and that final model selection should consider multiple metrics rather than a single optimisation score.